# 1장. 데이터베이스 연습 (SQL · 인덱스 · 트랜잭션 · MongoDB)

PDF 1장을 손으로 확인하는 노트북이에요. MySQL을 따로 설치하지 않아도 되게 파이썬 내장 **SQLite**로 연습해요.
`SELECT`·`JOIN`·`GROUP BY`·인덱스·트랜잭션 개념은 MySQL/PostgreSQL에서도 그대로 통해요.

**푸는 법**
1. 맨 위 셀부터 순서대로 실행해요.
2. `___` 자리를 채우고 실행 → 바로 아래 채점 셀을 실행해요 (✅/❌).
3. 막히면 `💡 정답 보기`를 열어요. 전체 정답은 `solutions/` 폴더에 있어요.

In [ ]:
# ▶ 먼저 이 셀을 실행하세요 (채점 도우미)
import os
STRICT = os.environ.get("PRACTICE_STRICT") == "1"
___ = None  # 빈칸 표시. 이 자리를 채우면 돼요

def check(name, cond, hint=""):
    """cond가 참이면 ✅, 아니면 ❌ (cond는 값이나 lambda)"""
    try:
        ok = bool(cond() if callable(cond) else cond)
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}"
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"  → {hint}"))
    if STRICT and not ok:
        raise AssertionError(name)

## 0. 실습용 DB 만들기 (그냥 실행)

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")          # 메모리 DB (노트북 끄면 사라짐)
conn.execute("PRAGMA foreign_keys = ON")    # SQLite는 FK 검사를 직접 켜야 해요
conn.executescript("""
CREATE TABLE users(
    id    INTEGER PRIMARY KEY,
    name  TEXT NOT NULL,
    point INTEGER NOT NULL DEFAULT 0 CHECK (point >= 0)
);
CREATE TABLE orders(
    id         INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL REFERENCES users(id),
    mall       TEXT,
    amount     INTEGER,
    status     TEXT,
    created_at TEXT
);
CREATE TABLE refunds(
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL REFERENCES orders(id),
    amount   INTEGER NOT NULL
);
""")
conn.executemany("INSERT INTO users VALUES (?,?,?)", [
    (1, "김철수", 1000), (2, "이영희", 0), (3, "박민수", 5000), (7, "최지우", 2000), (9, "정하늘", 0)])
conn.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", [
    (1001, 1, "A", 15000, "paid",      "2026-10-01 10:00"),
    (1002, 1, "B", 32000, "shipped",   "2026-10-02 11:30"),
    (1003, 7, "A", 29000, "paid",      "2026-10-03 09:10"),
    (1004, 3, "C",  8000, "paid",      "2026-10-03 15:00"),
    (1005, 7, "B", 41000, "shipped",   "2026-10-05 20:45"),
    (1006, 3, "A", 12000, "cancelled", "2026-10-06 08:00"),
    (1007, 1, "A", 27000, "paid",      "2026-10-07 13:20"),
])
conn.commit()

def q(sql, params=()):
    """SQL 실행 후 결과를 리스트로"""
    return conn.execute(sql, params).fetchall()

q("SELECT * FROM orders")

## 1-1. DB 종류 고르기
각 데이터에 가장 잘 맞는 DB 종류를 `"관계형"`, `"문서형"`, `"키-값"`, `"벡터DB"` 중에서 골라요.

In [ ]:
db_choice = {
    "주문·결제 (정확해야 하고 관계가 많음)": "관계형",
    "챗봇 대화 기록 (문서마다 필드가 달라짐)": "문서형",
    "로그인 세션·API 사용량 제한": "키-값",
    "RAG용 문서 임베딩 유사도 검색": "벡터DB",
}

In [ ]:
check("DB 종류", lambda: list(db_choice.values()) == ["관계형", "문서형", "키-값", "벡터DB"])

## 1-2. 기본키(PK)·외래키(FK) 직접 부딪혀 보기
아래 셀은 **일부러 실패하는** INSERT 두 개예요. 실행해서 어떤 에러가 나는지 보고 질문에 답해요.

In [ ]:
for sql in ["INSERT INTO orders VALUES (1001, 1, 'A', 1000, 'paid', '2026-10-08')",   # ① 이미 있는 id
            "INSERT INTO orders VALUES (2000, 999, 'A', 1000, 'paid', '2026-10-08')"]: # ② 없는 회원
    try:
        conn.execute(sql)
    except sqlite3.IntegrityError as e:
        print(type(e).__name__, "→", e)
conn.rollback()

In [ ]:
why_1 = "PK"   # 기본키는 중복 불가 (UNIQUE constraint failed)
why_2 = "FK"   # 없는 회원(999)의 주문은 외래키가 막음

In [ ]:
check("①은 기본키 중복", lambda: why_1 == "PK")
check("②는 외래키 위반", lambda: why_2 == "FK")

## 1-3. JOIN
### (1) INNER JOIN
모든 주문에 대해 `(주문id, 회원이름, 금액)`을 주문 id 순으로 뽑아요.

In [ ]:
sql_inner = """
SELECT o.id, u.name, o.amount
FROM orders o
JOIN users u ON o.user_id = u.id
ORDER BY o.id
"""
q(sql_inner)

In [ ]:
check("INNER JOIN", lambda: q(sql_inner) == [
    (1001, "김철수", 15000), (1002, "김철수", 32000), (1003, "최지우", 29000), (1004, "박민수", 8000),
    (1005, "최지우", 41000), (1006, "박민수", 12000), (1007, "김철수", 27000)])

### (2) LEFT JOIN
**주문이 없는 회원도 포함해서** 회원별 주문 수를 `(회원id, 이름, 주문수)`로 회원 id 순으로 뽑아요.
주문이 없으면 0이 나와야 해요.

> 힌트: `COUNT(*)`와 `COUNT(o.id)`의 차이를 생각해 보세요. LEFT JOIN에서 오른쪽이 없으면 `o.id`는 NULL이에요.

In [ ]:
sql_left = """
SELECT u.id, u.name, COUNT(o.id)
FROM users u
LEFT JOIN orders o ON o.user_id = u.id
GROUP BY u.id
ORDER BY u.id
"""
q(sql_left)   # COUNT(*)로 쓰면 주문 없는 회원도 1로 세져요

In [ ]:
check("LEFT JOIN + COUNT", lambda: q(sql_left) == [
    (1, "김철수", 3), (2, "이영희", 0), (3, "박민수", 2), (7, "최지우", 2), (9, "정하늘", 0)])

## 1-4. GROUP BY + HAVING
취소(`cancelled`)를 **뺀** 주문으로 몰(mall)별 매출 합계를 구하고, **합계가 40000 이상인 몰만** 합계 내림차순으로 뽑아요.
결과 모양: `[(몰, 합계), ...]`

> `WHERE`는 그룹을 만들기 **전** 행을 거르고, `HAVING`은 그룹을 만든 **후** 거른다는 차이가 핵심이에요.

In [ ]:
sql_group = """
SELECT mall, SUM(amount) AS total
FROM orders
WHERE status != 'cancelled'
GROUP BY mall
HAVING total >= 40000
ORDER BY total DESC
"""
q(sql_group)

In [ ]:
check("GROUP BY + HAVING", lambda: q(sql_group) == [("B", 73000), ("A", 71000)])

## 1-5. 인덱스 ★ 면접 단골
로그 2만 줄을 만들고, 인덱스 전/후의 **실행 계획**과 **속도**를 비교해요.
- `SCAN logs` = 처음부터 끝까지 다 읽음 (풀 스캔)
- `SEARCH logs USING INDEX ...` = 인덱스로 바로 찾아감

In [ ]:
import random, time
random.seed(0)
conn.execute("CREATE TABLE logs(id INTEGER PRIMARY KEY, user_id INTEGER, mall TEXT, created_at TEXT, msg TEXT)")
rows = []
for i in range(20000):
    day, hh, mm = random.randint(1, 30), random.randint(0, 23), random.randint(0, 59)
    rows.append((i, random.randint(1, 500), random.choice("ABCDE"),
                 f"2026-10-{day:02d} {hh:02d}:{mm:02d}", random.choice(["배송 문의", "환불 요청", "교환 문의", "결제 오류"])))
conn.executemany("INSERT INTO logs VALUES (?,?,?,?,?)", rows)
conn.commit()

def plan(sql):
    """EXPLAIN QUERY PLAN 결과를 한 줄 문자열로"""
    return " | ".join(r[-1] for r in q("EXPLAIN QUERY PLAN " + sql))

def bench(sql, n=300):
    t = time.perf_counter()
    for k in range(n):
        q(sql, (k % 500 + 1,))
    return time.perf_counter() - t

print(plan("SELECT * FROM logs WHERE user_id = 7"))
before = bench("SELECT * FROM logs WHERE user_id = ?")
print(f"인덱스 없음: {before:.3f}초")

### (1) `logs.user_id`에 `idx_logs_user` 인덱스를 만들어요

In [ ]:
conn.execute("CREATE INDEX idx_logs_user ON logs(user_id)")
print(plan("SELECT * FROM logs WHERE user_id = 7"))
after = bench("SELECT * FROM logs WHERE user_id = ?")
print(f"인덱스 있음: {after:.3f}초  ({before/after:.0f}배 빠름)")

In [ ]:
check("user_id 인덱스 사용", lambda: "USING INDEX idx_logs_user" in plan("SELECT * FROM logs WHERE user_id = 7"))
check("더 빨라졌다", lambda: after < before)

### (2) 복합 인덱스는 **왼쪽 열부터**
`(mall, created_at)` 복합 인덱스를 만든 뒤, 아래 세 쿼리가 인덱스로 **SEARCH** 하는지(True) 아닌지(False) **먼저 예상**해서 적고, 채점 셀로 확인해요.

In [ ]:
conn.execute("CREATE INDEX idx_logs_mall_time ON logs(mall, created_at)")
queries = {
    "q1": "SELECT * FROM logs WHERE mall = 'A' AND created_at >= '2026-10-20'",
    "q2": "SELECT * FROM logs WHERE mall = 'A'",
    "q3": "SELECT * FROM logs WHERE created_at >= '2026-10-20'",
}

In [ ]:
my_guess = {"q1": True, "q2": True, "q3": False}   # q3은 왼쪽 열(mall)이 없어서 못 씀

In [ ]:
actual = {k: "SEARCH" in plan(v) for k, v in queries.items()}
for k, v in queries.items():
    print(k, "→", plan(v))
check("복합 인덱스 예상", lambda: my_guess == actual, f"실제: {actual}")

### (3) 인덱스가 안 먹는 쿼리 고치기
`created_at`에만 인덱스를 하나 더 만들었어요. 그런데 열에 **함수**(`DATE()`)를 씌우면 인덱스를 못 써요.
같은 결과(10월 15일 로그 수)를 내면서 인덱스를 쓰도록 **범위 조건**으로 다시 써요.

> 계획에 `SCAN ... USING COVERING INDEX`가 보이면 인덱스를 **처음부터 끝까지 훑는** 것(풀 스캔)이에요. 목표는 `SEARCH`예요.

In [ ]:
conn.execute("CREATE INDEX idx_logs_time ON logs(created_at)")
bad_sql = "SELECT COUNT(*) FROM logs WHERE DATE(created_at) = '2026-10-15'"
print(plan(bad_sql), "→", q(bad_sql))

In [ ]:
good_sql = "SELECT COUNT(*) FROM logs WHERE created_at >= '2026-10-15' AND created_at < '2026-10-16'"
print(plan(good_sql), "→", q(good_sql))

In [ ]:
check("결과가 같다", lambda: q(good_sql) == q(bad_sql))
check("인덱스로 SEARCH 한다", lambda: "SEARCH" in plan(good_sql) and "idx_logs_time" in plan(good_sql),
      plan(good_sql) if good_sql else "")

In [ ]:
idx_quiz = {
    "인덱스를 많이 만들수록 INSERT도 빨라진다": False,   # 쓰기마다 인덱스도 고쳐야 해서 느려짐
    "성별(2~3가지 값) 열은 인덱스 효과가 작다": True,
    "LIKE '%배송' 처럼 앞에 %가 붙으면 보통 인덱스를 못 쓴다": True,
    "인덱스는 보통 B-Tree 구조다": True,
}

In [ ]:
check("인덱스 OX", lambda: list(idx_quiz.values()) == [False, True, True, True])

## 1-6. 트랜잭션과 ACID ★
환불 = ① 주문 상태 변경 + ② 포인트 복구 + ③ 환불 기록. **셋 다 되거나 셋 다 안 돼야** 해요.

Python `sqlite3`에서는 `with conn:` 블록이 트랜잭션이에요. 블록이 정상 종료되면 `COMMIT`, 예외가 나면 `ROLLBACK` 해요.
`refund()`를 완성해요.

In [ ]:
def refund(order_id, amount, point_back):
    with conn:   # 여기 안의 SQL은 전부 성공하거나 전부 취소
        conn.execute("UPDATE orders SET status = 'refunded' WHERE id = ?", (order_id,))
        conn.execute("UPDATE users SET point = point + ? WHERE id = (SELECT user_id FROM orders WHERE id = ?)",
                     (point_back, order_id))
        conn.execute("INSERT INTO refunds(order_id, amount) VALUES (?, ?)", (order_id, amount))

**성공 케이스**: 1003번 주문 환불 (회원 7, 포인트 2000 → 5000)

In [ ]:
refund(1003, 29000, 3000)
check("주문 상태 refunded", lambda: q("SELECT status FROM orders WHERE id=1003") == [("refunded",)])
check("포인트 복구", lambda: q("SELECT point FROM users WHERE id=7") == [(5000,)])
check("환불 기록 1건", lambda: q("SELECT order_id, amount FROM refunds") == [(1003, 29000)])

**실패 케이스**: `amount=None`이면 ③에서 `NOT NULL` 위반으로 실패해요.
①②는 이미 실행됐지만 **롤백**되어 아무것도 바뀌지 않아야 해요 (원자성).

In [ ]:
try:
    refund(1005, None, 3000)
except sqlite3.IntegrityError as e:
    print("실패:", e)
check("1005 상태 그대로 shipped", lambda: q("SELECT status FROM orders WHERE id=1005") == [("shipped",)])
check("포인트 그대로 5000", lambda: q("SELECT point FROM users WHERE id=7") == [(5000,)])
check("환불 기록 그대로 1건", lambda: q("SELECT COUNT(*) FROM refunds") == [(1,)])

### 일관성(C) 체험: `CHECK (point >= 0)`
포인트를 음수로 만들려고 하면 DB가 막아요. 아래 `try` 안을 실행해 보고, 어떤 ACID 성질인지 답해요.

In [ ]:
try:
    with conn:
        conn.execute("UPDATE users SET point = point - 999999 WHERE id = 7")
except sqlite3.IntegrityError as e:
    print("막힘:", e)
print(q("SELECT point FROM users WHERE id=7"))

In [ ]:
acid = {
    "환불 3단계 중 하나가 실패하면 셋 다 되돌린다": "A",
    "포인트가 음수가 되는 상태로는 끝나지 않는다": "C",
    "다른 트랜잭션의 커밋 안 된 중간값을 보지 않는다": "I",
    "COMMIT 직후 서버 전원이 나가도 결과가 남아 있다": "D",
}
isolation = {
    "MySQL InnoDB 기본": "REPEATABLE READ",
    "PostgreSQL 기본": "READ COMMITTED",
}
anomaly = {
    "커밋 안 된 값을 읽었는데 그 트랜잭션이 롤백됨": "Dirty Read",
    "같은 행을 두 번 읽었는데 값이 다름": "Non-repeatable Read",
    "같은 조건으로 두 번 셌는데 행 개수가 다름": "Phantom Read",
}

In [ ]:
check("ACID", lambda: list(acid.values()) == ["A", "C", "I", "D"])
check("격리 수준 기본값", lambda: list(isolation.values()) == ["REPEATABLE READ", "READ COMMITTED"])
check("동시성 문제 이름", lambda: list(anomaly.values()) == ["Dirty Read", "Non-repeatable Read", "Phantom Read"])

## 1-7. MySQL · MariaDB · PostgreSQL 문법 차이

In [ ]:
dialect = {
    "MySQL 자동 증가 PK":        "id INT AUTO_INCREMENT PRIMARY KEY",
    "PostgreSQL 자동 증가 PK":   "id SERIAL PRIMARY KEY",
    "MySQL 문자열 붙이기":        "CONCAT(a, b)",
    "PostgreSQL 문자열 붙이기":   "a || b",
    "MySQL 테이블 목록":          "SHOW TABLES;",
    "PostgreSQL(psql) 테이블 목록": "\\dt",
    "이모지까지 넣으려면 MySQL 문자셋": "utf8mb4",
}

In [ ]:
norm = lambda s: str(s).replace(" ", "").lower().rstrip(";")
check("PostgreSQL 자동 증가", lambda: "serial" in norm(dialect["PostgreSQL 자동 증가 PK"]) or "identity" in norm(dialect["PostgreSQL 자동 증가 PK"]))
check("MySQL 자동 증가", lambda: "auto_increment" in norm(dialect["MySQL 자동 증가 PK"]))
check("PostgreSQL 문자열 붙이기", lambda: "||" in dialect["PostgreSQL 문자열 붙이기"])
check("psql 테이블 목록", lambda: norm(dialect["PostgreSQL(psql) 테이블 목록"]) == "\\dt")
check("utf8mb4", lambda: norm(dialect["이모지까지 넣으려면 MySQL 문자셋"]) == "utf8mb4")

## 1-8. MongoDB (문서형)
MongoDB 서버 없이 연습하려고, 아래에 **Mongo 필터 문법을 흉내 내는 작은 함수** `mfind`를 만들어 뒀어요.
(`$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$ne` 지원) 실제 `db.chats.find(필터)`에 넣는 dict와 똑같이 쓰면 돼요.

In [ ]:
chats = [
    {"_id": 1, "mall": "A", "user": 7, "turns": 2, "handoff": False, "tags": ["배송"]},
    {"_id": 2, "mall": "A", "user": 3, "turns": 5, "handoff": True,  "tags": ["환불"]},
    {"_id": 3, "mall": "B", "user": 7, "turns": 4, "handoff": True,  "tags": ["교환", "배송"]},
    {"_id": 4, "mall": "A", "user": 1, "turns": 1, "handoff": False},           # tags 필드 없음 (문서형이라 OK)
    {"_id": 5, "mall": "C", "user": 7, "turns": 6, "handoff": False, "tags": ["결제"]},
    {"_id": 6, "mall": "A", "user": 9, "turns": 3, "handoff": True,  "tags": ["환불"], "agent": "kim"},
]

_OPS = {"$gt": lambda a, b: a is not None and a > b, "$gte": lambda a, b: a is not None and a >= b,
        "$lt": lambda a, b: a is not None and a < b,  "$lte": lambda a, b: a is not None and a <= b,
        "$in": lambda a, b: a in b, "$ne": lambda a, b: a != b}

def _match(doc, flt):
    for field, cond in flt.items():
        val = doc.get(field)
        if isinstance(cond, dict):
            if not all(_OPS[op](val, arg) for op, arg in cond.items()):
                return False
        elif isinstance(val, list):
            if cond not in val:          # 배열 필드는 "포함"이면 매치 (Mongo와 같음)
                return False
        elif val != cond:
            return False
    return True

def mfind(coll, flt, sort=None, limit=None):
    """sort=("_id", -1) 처럼"""
    out = [d for d in coll if _match(d, flt)]
    if sort:
        out.sort(key=lambda d: d.get(sort[0]), reverse=sort[1] == -1)
    return out[:limit] if limit else out

[d["_id"] for d in mfind(chats, {"mall": "A"})]

### (1) 필터 dict 쓰기
- `f1`: 몰 A에서 상담사로 넘어간(`handoff: True`) 대화
- `f2`: 회원 7의 대화 중 `turns`가 **3 이상**
- `f3`: `tags`에 `"배송"`이 들어간 대화
- `f4`: 몰이 A 또는 C인 대화 (`$in`)

In [ ]:
f1 = {"mall": "A", "handoff": True}
f2 = {"user": 7, "turns": {"$gte": 3}}
f3 = {"tags": "배송"}
f4 = {"mall": {"$in": ["A", "C"]}}

In [ ]:
ids = lambda f, **kw: [d["_id"] for d in mfind(chats, f, **kw)]
check("f1", lambda: ids(f1) == [2, 6])
check("f2", lambda: ids(f2) == [3, 5])
check("f3", lambda: ids(f3) == [1, 3])
check("f4", lambda: ids(f4) == [1, 2, 4, 5, 6])

### (2) `.sort({_id: -1}).limit(2)` + `aggregate($group)` 흉내내기
- `latest`: f1 결과를 `_id` 내림차순으로 2개 → `mfind(..., sort=..., limit=...)`
- `by_mall`: `db.chats.aggregate([{ $group: { _id: "$mall", cnt: { $sum: 1 } } }])` 와 같은 결과를 `{"A": 4, ...}` dict로 (= SQL `GROUP BY`)

In [ ]:
latest = mfind(chats, f1, sort=("_id", -1), limit=2)

by_mall = {}
for d in chats:
    by_mall[d["mall"]] = by_mall.get(d["mall"], 0) + 1
by_mall

In [ ]:
check("sort + limit", lambda: [d["_id"] for d in latest] == [6, 2])
check("$group 집계", lambda: by_mall == {"A": 4, "B": 1, "C": 1})

### (3) SQL ↔ MongoDB 용어 짝짓기

In [ ]:
sql_to_mongo = {
    "table":    "collection",
    "row":      "document",
    "column":   "field",
    "GROUP BY": "$group",
    "JOIN":     "$lookup",
}

In [ ]:
check("용어 짝", lambda: sql_to_mongo == {"table": "collection", "row": "document", "column": "field",
                                         "GROUP BY": "$group", "JOIN": "$lookup"})

## ✍️ 말로 설명해 보기 (30초)
1. 인덱스는 왜 빠르고, 언제 오히려 손해인가?
2. 트랜잭션과 ACID를 환불 예시로 설명하면?
3. MySQL과 MariaDB의 관계, RDB와 MongoDB는 언제 각각 쓰나?

<details><summary>모범 답안</summary>

1. 인덱스는 특정 열 값을 정렬해 둔 B-Tree라서 풀 스캔 없이 바로 찾아가요. 대신 INSERT/UPDATE마다 인덱스도 고쳐야 해서 쓰기가 느려지고 공간을 더 써요. 값 종류가 적은 열(성별)이나, 함수를 씌운 조건·앞 `%` LIKE에서는 효과가 없어요.
2. 환불은 주문 상태 변경·포인트 복구·환불 기록 세 개를 한 트랜잭션으로 묶어요. 하나라도 실패하면 전부 롤백(원자성), 포인트가 음수가 되는 상태로 끝나지 않음(일관성), 동시에 도는 다른 환불이 중간 상태를 못 봄(격리성), 커밋하면 서버가 꺼져도 남음(지속성).
3. MariaDB는 MySQL 원개발자들이 갈라져 나와 만든 오픈소스로 거의 호환돼요. 주문·결제처럼 스키마가 정해지고 정확성이 중요한 건 RDB, 대화 로그·상품 속성처럼 구조가 자주 바뀌는 건 MongoDB 같은 문서형이 맞아요.
</details>